In [ ]:
!pip install PyPortfolioOpt
!pip install yfinance

In [1]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import numpy as np

from pypfopt import black_litterman, risk_models
from pypfopt import BlackLittermanModel, plotting

import pypfopt

In [3]:
tickers = ['ITSA4.SA', 'BBAS3.SA', 'MULT3.SA', 'WEGE3.SA' , 'CAML3.SA']
prices = yf.download(tickers, period='1y' ,auto_adjust=True)['Close']
prices.head()

[*********************100%***********************]  5 of 5 completed


Ticker,BBAS3.SA,CAML3.SA,ITSA4.SA,MULT3.SA,WEGE3.SA
Date,,,,,
2025-07-24,19.609949,4.609305,9.132494,24.381123,34.545227
2025-07-25,19.776054,4.599801,9.167620,24.284943,34.488018
2025-07-28,19.482929,4.200645,9.000776,23.890614,35.188778
2025-07-29,19.492704,4.238660,9.027120,24.275324,35.045074
2025-07-30,19.443848,4.371712,9.132494,24.477301,35.303741


In [5]:
market_prices = yf.download('^BVSP', period='1y',auto_adjust=True)['Close']
market_prices.head()

[*********************100%***********************]  1 of 1 completed


Ticker,^BVSP
Date,
2025-07-24,133808.0
2025-07-25,133524.0
2025-07-28,132129.0
2025-07-29,132726.0
2025-07-30,133990.0


In [6]:
mktcaps = {}
for t in tickers:
    stock = yf.Ticker(t)
    mktcaps[t] = stock.info["marketCap"]
mktcaps

{'ITSA4.SA': 150738862080,
 'BBAS3.SA': 116175568896,
 'MULT3.SA': 13768946688,
 'WEGE3.SA': 192966213632,
 'CAML3.SA': 1524588288}

In [7]:
# Taxa de livre de risco
Rf = 0.1425

In [8]:
S = risk_models.sample_cov(prices)
delta = black_litterman.market_implied_risk_aversion(market_prices, risk_free_rate=Rf)
delta

4.701855403634288

In [10]:
market_prior = black_litterman.market_implied_prior_returns(mktcaps, delta, S, risk_free_rate=Rf)
market_prior

Ticker
BBAS3.SA    0.367448
CAML3.SA    0.341071
ITSA4.SA    0.334686
MULT3.SA    0.319879
WEGE3.SA    0.384644
dtype: float64

In [13]:
# Não é necessário fornecer para todos os ativos
viewdict = {
    "BBAS3.SA": 0.30,
    "MULT3.SA": 0.10,
  
}

bl = BlackLittermanModel(S, pi=market_prior, absolute_views=viewdict)

In [14]:
mu = bl.bl_returns()
mu

Ticker
BBAS3.SA    0.297580
CAML3.SA    0.246270
ITSA4.SA    0.261599
MULT3.SA    0.210597
WEGE3.SA    0.348852
dtype: float64

In [15]:
pd.concat([market_prior, mu, pd.Series(viewdict)],
          axis=1,
          keys=['a_priori', 'a_posteriori', 'visões'])

,a_priori,a_posteriori,visões
BBAS3.SA,0.367448,0.297580,0.3
CAML3.SA,0.341071,0.246270,NaN
ITSA4.SA,0.334686,0.261599,NaN
MULT3.SA,0.319879,0.210597,0.1
WEGE3.SA,0.384644,0.348852,NaN


In [16]:
cov = bl.bl_cov()
cov

Ticker,BBAS3.SA,CAML3.SA,ITSA4.SA,MULT3.SA,WEGE3.SA
Ticker,,,,,
BBAS3.SA,0.086030,0.050344,0.044771,0.046346,0.029737
CAML3.SA,0.050344,0.188432,0.052251,0.062158,0.029280
ITSA4.SA,0.044771,0.052251,0.058103,0.047986,0.027128
MULT3.SA,0.046346,0.062158,0.047986,0.071525,0.023585
WEGE3.SA,0.029737,0.029280,0.027128,0.023585,0.090763


In [17]:
from pypfopt.efficient_frontier import EfficientFrontier
ef_sr = EfficientFrontier(mu, cov)

In [18]:
ef_sr.max_sharpe()

OrderedDict([('BBAS3.SA', 0.221529333383861),
             ('CAML3.SA', 0.0),
             ('ITSA4.SA', 0.3416768171827271),
             ('MULT3.SA', 0.0),
             ('WEGE3.SA', 0.436793849433412)])

In [19]:
ef_sr.clean_weights()

OrderedDict([('BBAS3.SA', 0.22153),
             ('CAML3.SA', 0.0),
             ('ITSA4.SA', 0.34168),
             ('MULT3.SA', 0.0),
             ('WEGE3.SA', 0.43679)])

In [20]:
ef_sr.portfolio_performance(verbose=True)

Expected annual return: 30.8%
Annual volatility: 22.1%
Sharpe Ratio: 1.39


(0.30768124517041173, 0.22124989243256368, 1.3906503717925742)

In [21]:
ef_mv = EfficientFrontier(mu, cov)

In [ ]:
ef_mv.min_volatility()

ef_mv.clean_weights()

In [23]:
ef_mv.portfolio_performance(verbose=True)

ValueError: Weights is None

In [24]:
confidences = [
    0.6,
    0.4,
    0.5,
    0.5,
    0.5
]

In [25]:
market_prior

Ticker
BBAS3.SA    0.367448
CAML3.SA    0.341071
ITSA4.SA    0.334686
MULT3.SA    0.319879
WEGE3.SA    0.384644
dtype: float64

In [26]:
bl = BlackLittermanModel(S, pi=market_prior,
                         absolute_views=viewdict,
                         omega="idzorek",
                         view_confidences=confidences)

ValueError: view_confidences should be a numpy 1D array or vector with the same length as the number of views.

In [ ]:
colunas = prices.columns.to_list()
#colunas.append("q")
print(colunas)

#visoes = pd.DataFrame(columns=colunas)

visao1 = {'B3SA3.SA': 1}
visao2 = {'MGLU3.SA': 1}
visao3 = {
    'BBDC4.SA': -0.5,
    'ITUB4.SA': -0.5,
    'PETR4.SA': 0.5,
    'VALE3.SA': 0.5,
}

In [ ]:
# Não é necessário fornecer para todos os ativos
viewdict = {
    "B3SA3.SA": 0.30,
    "MGLU3.SA": 0.10,
    "VALE3.SA": 0.5,
    "BBDC4.SA": 0.25
}

In [ ]:
confidences = [
    0.6,
    0.4,
    0.5,
    0.5
]

In [ ]:
bl = BlackLittermanModel(S, pi=market_prior,
                         absolute_views=viewdict,
                         omega="idzorek",
                         view_confidences=confidences)

In [ ]:
colunas = prices.columns.to_list()
#colunas.append("q")
print(colunas)

#visoes = pd.DataFrame(columns=colunas)

visao1 = {'B3SA3.SA': 1}
visao2 = {'MGLU3.SA': 1}
visao3 = {
    'BBDC4.SA': -0.5,
    'ITUB4.SA': -0.5,
    'PETR4.SA': 0.5,
    'VALE3.SA': 0.5
}

In [ ]:
colunas = prices.columns.to_list()
colunas.append("q")
print(colunas)

visoes = pd.DataFrame(columns=colunas)

visao1 = {'B3SA3.SA': 1}
visao2 = {'MGLU3.SA': 1}
visao3 = {
    'BBDC4.SA': -0.5,
    'ITUB4.SA': -0.5,
    'PETR4.SA': 0.5,
    'VALE3.SA': 0.5
}

# Using pd.concat instead of the deprecated append method
visoes = pd.concat([visoes, pd.DataFrame([visao1])], ignore_index=True)
visoes = pd.concat([visoes, pd.DataFrame([visao2])], ignore_index=True)
visoes = pd.concat([visoes, pd.DataFrame([visao3])], ignore_index=True)
visoes.fillna(0, inplace=True)
visoes['Q'] = [0.50, -0.10, 0.30]

visoes

In [ ]:
bl_pq = BlackLittermanModel(S, pi=market_prior,
                            P=visoes[prices.columns].values,
                            Q=visoes['Q'].values.reshape(-1, 1)
                            )

In [ ]:
mu = bl_pq.bl_returns()
mu

In [ ]:
pd.concat([market_prior, mu],
          axis=1,
          keys=['a_priori', 'a_posteriori'])

In [ ]:
cov = bl_pq.bl_cov()
cov

In [ ]:
from pypopfopt.efficient_frontier import EfficientFrontier
ef_sr = EfficientFrontier(mu, cov, weight_bounds=(-1,1))

In [ ]:
ef_sr.max_sharpe()

In [ ]:
ef_sr.clean_weights()

In [ ]:
ef_sr.portfolio_performance(verbose=True);